# RealDrift: generation and drift detection for BPIC2012
Run the cells top to bottom. Each step calls the same scripts as the command line, so results are identical.
Set `QUICK = True` for a smoke test on a few cases per instance (minutes, not hours).

In [ ]:
import os, sys, subprocess, json, glob
import pandas as pd
ROOT = os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
os.chdir(ROOT); sys.path.insert(0, ROOT)

CONFIG   = "configs/bpic2012_config.yaml"
DATASET  = "bpic12"
QUICK    = False          # True, only for a smoke test
QUICK_MAX_CASES = 150
SINGLE   = dict(tier="sudden", seed=0)   # used by section 3

def run(cmd):
    print("$", " ".join(cmd))
    p = subprocess.run(cmd, cwd=ROOT, text=True, capture_output=True)
    out = "\n".join(l for l in (p.stdout + p.stderr).splitlines() if "it/s" not in l and "kde" not in l)
    print(out[-4000:])
    if p.returncode != 0:
        raise RuntimeError(f"command failed with exit code {p.returncode}")
quick = ["--max-cases-per-instance", str(QUICK_MAX_CASES)] if QUICK else []

## 1. Check the inputs

In [ ]:
import yaml
cfg = yaml.safe_load(open(CONFIG))
print("n_clusters", cfg["n_clusters"], "| d_supp", cfg["calendar_params"]["d_supp"],
      "| SA iterations", cfg["composer"]["sa_iterations"])
for d in ["data/sublogs", "data/concept_pools", "data/calendars"]:
    print(d, sorted(os.listdir(d)))

## 2. Optional preparation (only when starting from the raw log)
Rebuilds clusters, sublogs and calendars from `data/raw`. The existing files are reproduced exactly for BPIC12.
Delete `data/calendars/*.pkl` first when `d_supp` was changed.

In [ ]:
RUN_PREPARE = False
if RUN_PREPARE:
    run([sys.executable, "scripts/prepare_dataset.py", "--config", CONFIG, "--silhouette-report"])

In [ ]:
RUN_CALENDAR_DIAGNOSTICS = False   # how many real cases fit the calendars per d_supp
if RUN_CALENDAR_DIAGNOSTICS:
    run([sys.executable, "scripts/calendar_diagnostics.py", "--config", CONFIG])
    display(pd.read_csv(f"outputs/{DATASET}_calendar_diagnostics.csv"))

## 3. Generate one stream

In [ ]:
out = "outputs/single"
run([sys.executable, "generate_drift_log.py", "--config", CONFIG, "--tier", SINGLE["tier"],
     "--seed", str(SINGLE["seed"]), "--out-dir", out, "--export-xes"] + quick)
prefix = f"{out}/{DATASET}_{SINGLE['tier']}_seed{SINGLE['seed']}"
display(pd.read_csv(prefix + "_transitions.csv"))
m = json.load(open(prefix + "_manifest.json"))
print({k: m[k] for k in ["n_placed", "n_excluded", "mean_delay_per_case_h"]},
      "validation", m["validation"]["ok"], "ground truth", m["ground_truth_check"]["ok"])

In [ ]:
log = pd.read_csv(prefix + "_drift_log.csv")
tr = pd.read_csv(prefix + "_transitions.csv")
ts_col = [c for c in log.columns if "time" in c.lower()][0]
log["_t"] = pd.to_datetime(log[ts_col], format="mixed", utc=True)
import matplotlib.pyplot as plt
weekly = log.set_index("_t").resample("W").size()
ax = weekly.plot(figsize=(10, 3), title="Events per week with ground-truth drifts")
for _, r in tr.iterrows():
    s, e = pd.to_datetime(r["start_ts"], utc=True), pd.to_datetime(r["end_ts"], utc=True)
    ax.axvspan(s, e, color="red", alpha=0.3) if e > s else ax.axvline(s, color="red")
plt.show()

## 4. Generate all streams
Evaluation seeds 0 to 4 and tuning seeds 100 to 102 (see `configs/experiment.yaml`). Finished runs are reused, set `FORCE = True` to redo them.
Output goes to `outputs/streams/eval` and `outputs/streams/tune`.

In [ ]:
FORCE = False
for phase in ["eval", "tune"]:
    run([sys.executable, "scripts/run_composition.py", "--datasets", DATASET, "--phase", phase]
        + (["--force"] if FORCE else []) + quick)

In [ ]:
summ = glob.glob("outputs/**/composition_summary.csv", recursive=True)
print(summ)
for f in summ:
    df = pd.read_csv(f)
    display(df[df["dataset"] == DATASET])

## 5. Drift detection
Detectors are added as modules that register themselves (see `evaluation/detectors.py`) and are listed in `configs/detectors.yaml`.
The cell below stops with a clear message while no detector is configured.

In [ ]:
det = yaml.safe_load(open("configs/detectors.yaml")).get("detectors") or {}
print("configured detectors:", list(det))
assert det, "No detector configured yet. Add evaluation/detector_pelt.py and detector_vc.py and list them in configs/detectors.yaml."

In [ ]:
run([sys.executable, "evaluation/run_detection.py", "--phase", "tune"])
run([sys.executable, "evaluation/run_detection.py", "--phase", "eval"])

## 6. Results (mean and sample standard deviation over seeds)

In [ ]:
s = pd.read_csv("outputs/detection_summary.csv")

display(s[s["tolerance"] == "main"].round(3))

## 7. Export for the continual learning evaluation (COMPASS)

In [ ]:
os.makedirs("outputs/cl", exist_ok=True)
for tier in ["sudden", "gradual", "hard", "recurrent"]:
    p = f"outputs/streams/eval/{DATASET}_{tier}_seed0"
    if os.path.exists(p + "_drift_log.csv"):
        run([sys.executable, "evaluation/export_cl_tasks.py", p, "--out", f"outputs/cl/{DATASET}_{tier}_seed0.csv"])